In [2]:
import pandas as pd
import joblib
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import warnings
warnings.filterwarnings('ignore')

In [3]:
print("Loading trained model and vectorizer...")
model = joblib.load('C:/Users/Stefan/SBC/Hate_Speech_Recon/Models/Saved_models/best_model.pkl')
vectorizer = joblib.load('C:/Users/Stefan/SBC/Hate_Speech_Recon/Models/Saved_models/tfidf_vectorizer.pkl')
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
print("Model and vectorizer loaded successfully!")


Loading trained model and vectorizer...
Model and vectorizer loaded successfully!


In [4]:
def preprocess_tweet(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'&\w+;', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    try:
        tokens = word_tokenize(text)
    except:
        tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

print("Preprocessing function defined")

Preprocessing function defined


In [5]:
def predict_tweet(tweet, show_confidence=True):
    cleaned = preprocess_tweet(tweet)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    
    class_map = {
        0: 'Hate Speech',
        1: 'Offensive Language',
        2: 'Neither'
    }
    class_name = class_map[prediction]
    confidence = None
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(vectorized)[0]
        confidence = max(proba)
    print(f"\n{'='*70}")
    print(f"Original Tweet: {tweet}")
    print(f"Cleaned Tweet:  {cleaned}")
    print(f"-"*70)
    print(f"Prediction: {class_name}")
    if confidence:
        print(f"Confidence: {confidence:.2%}")
    print(f"{'='*70}")
    return prediction, class_name, confidence

print("Prediction function ready")

Prediction function ready


In [6]:
print("\n" + "="*70)
print("TESTING ON SAMPLE TWEETS")
print("="*70)
sample_tweets = [
    "I love spending time with my family on weekends!",
    "This is absolutely terrible and disgusting behavior",
    "You're an idiot and I hate you",
    "Check out this awesome new restaurant downtown",
    "Can't believe this is happening, so frustrating!"
]
results = []
for i, tweet in enumerate(sample_tweets, 1):
    print(f"\n--- Sample {i} ---")
    pred, class_name, conf = predict_tweet(tweet)
    results.append({
        'tweet': tweet,
        'prediction': class_name,
        'confidence': conf if conf else 'N/A'
    })
print("\n" + "="*70)
print("PREDICTION SUMMARY")
print("="*70)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))



TESTING ON SAMPLE TWEETS

--- Sample 1 ---

Original Tweet: I love spending time with my family on weekends!
Cleaned Tweet:  love spending time family weekend
----------------------------------------------------------------------
Prediction: Offensive Language

--- Sample 2 ---

Original Tweet: This is absolutely terrible and disgusting behavior
Cleaned Tweet:  absolutely terrible disgusting behavior
----------------------------------------------------------------------
Prediction: Neither

--- Sample 3 ---

Original Tweet: You're an idiot and I hate you
Cleaned Tweet:  youre idiot hate
----------------------------------------------------------------------
Prediction: Hate Speech

--- Sample 4 ---

Original Tweet: Check out this awesome new restaurant downtown
Cleaned Tweet:  check awesome new restaurant downtown
----------------------------------------------------------------------
Prediction: Neither

--- Sample 5 ---

Original Tweet: Can't believe this is happening, so frustrating!

In [7]:
def moderate_content(tweet, threshold=0.7):
    cleaned = preprocess_tweet(tweet)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    class_map = {0: 'Hate Speech', 1: 'Offensive Language', 2: 'Neither'}
    class_name = class_map[prediction]
    confidence = None
    action = 'ALLOW'
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(vectorized)[0]
        confidence = max(proba)
        if prediction in [0, 1]:  # Hate speech or offensive
            if confidence >= threshold:
                action = 'FLAG'
            else:
                action = 'REVIEW'
        else: 
            action = 'ALLOW'
    else:
        if prediction in [0, 1]:
            action = 'FLAG'
        else:
            action = 'ALLOW'
    
    print(f"\n{'='*70}")
    print(f"CONTENT MODERATION REPORT")
    print(f"{'='*70}")
    print(f"Tweet: {tweet}")
    print(f"-"*70)
    print(f"Classification: {class_name}")
    print(f"Confidence: {confidence:.2%}" if confidence else "Confidence: N/A")
    print(f"Action: {action}")
    print(f"{'='*70}")
    return action, class_name, confidence

print("\n" + "="*70)
print("CONTENT MODERATION SIMULATOR")
print("="*70)


CONTENT MODERATION SIMULATOR


In [8]:
mod_tweets = [
    "Have a wonderful day everyone!",
    "This is complete shit and waste of time",
    "You're the worst person ever and everyone hates you"
]

for tweet in mod_tweets:
    moderate_content(tweet, threshold=0.7)


CONTENT MODERATION REPORT
Tweet: Have a wonderful day everyone!
----------------------------------------------------------------------
Classification: Neither
Confidence: N/A
Action: ALLOW

CONTENT MODERATION REPORT
Tweet: This is complete shit and waste of time
----------------------------------------------------------------------
Classification: Offensive Language
Confidence: N/A
Action: FLAG

CONTENT MODERATION REPORT
Tweet: You're the worst person ever and everyone hates you
----------------------------------------------------------------------
Classification: Hate Speech
Confidence: N/A
Action: FLAG
